# Intuition test: deep learning foundations

Questions come first. For each one, write your prediction in the cell marked `Prediction`, then run the code cell below it. Answers are at the end, after every question. Do not scroll down early.

Covers notebooks 01 to 07. Seeded and self-contained.

In [ ]:
from common.dl import set_seed, setup_keras, weight_gradient_norms

setup_keras("torch")
import keras
import numpy as np
import torch

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## Q1. XOR

Four points: (0,0) and (1,1) are class 0, (0,1) and (1,0) are class 1. Can a single layer solve it? A network with one hidden layer of 4 tanh units, trained for 500 full-batch steps? Predict the accuracy of each.

Prediction: *(write here)*

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype="float32")
y = np.array([0, 1, 1, 0])
for name, hidden in [
    ("no hidden layer", []),
    ("4 tanh units", [keras.layers.Dense(4, activation="tanh")]),
]:
    set_seed(SEED)
    net = keras.Sequential(
        [keras.Input((2,)), *hidden, keras.layers.Dense(1, activation="sigmoid")]
    )
    net.compile(
        optimizer=keras.optimizers.Adam(0.05), loss="binary_crossentropy", metrics=["accuracy"]
    )
    net.fit(X, y, epochs=500, batch_size=4, verbose=0)
    print(f"{name}: accuracy {net.evaluate(X, y, verbose=0)[1]:.2f}")

## Q2. Ten sigmoid layers

A network of 10 sigmoid layers of 32 units, with PyTorch's default initialisation, on random data. What is the ratio of the gradient norm at the first layer to the gradient norm at the last layer? Within a factor of 10 of 1, or far from it? Compare with the same net using ReLU.

Prediction: *(write here)*

In [ ]:
data = rng.normal(size=(256, 20)).astype("float32")
target = rng.integers(0, 2, 256)
for act in (torch.nn.Sigmoid, torch.nn.ReLU):
    torch.manual_seed(SEED)
    layers, n_in = [], 20
    for _ in range(10):
        layers += [torch.nn.Linear(n_in, 32), act()]
        n_in = 32
    net = torch.nn.Sequential(*layers, torch.nn.Linear(32, 2))
    norms = list(
        weight_gradient_norms(net, data, target, torch.nn.functional.cross_entropy).values()
    )
    print(
        f"{act.__name__}: first {norms[0]:.1e}, last {norms[-1]:.1e}, ratio {norms[0] / norms[-1]:.1e}"
    )

## Q3. SGD against Adam on a stretched bowl

The loss is $J=\tfrac12(x^2+100y^2)$, starting at (1, 1). The curvature is 100 times higher in $y$. SGD uses the largest stable rate, 0.019. Adam uses 0.1. After 100 steps, which has the lower loss, and by how many orders of magnitude?

Prediction: *(write here)*

In [ ]:
def run(use_adam: bool, steps: int = 100) -> float:
    p, m1, s, lr = np.array([1.0, 1.0]), 0.0, 0.0, 0.1 if use_adam else 0.019
    for t in range(1, steps + 1):
        g = np.array([p[0], 100 * p[1]])
        if use_adam:
            m1, s = 0.9 * m1 + 0.1 * g, 0.999 * s + 0.001 * g**2
            p = p - lr * (m1 / (1 - 0.9**t)) / (np.sqrt(s / (1 - 0.999**t)) + 1e-8)
        else:
            p = p - lr * g
    return 0.5 * (p[0] ** 2 + 100 * p[1] ** 2)


print(f"SGD loss {run(False):.2e}, Adam loss {run(True):.2e}")

## Q4. Schedule values

A cosine schedule from 0.01 over 1,000 steps. What is the rate at step 500? At step 900? What fraction of the total learning-rate "area" (the sum of the rate over all steps) comes in the first half of training?

Prediction: *(write here)*

In [ ]:
t = np.arange(1000)
lr = 0.5 * 0.01 * (1 + np.cos(np.pi * t / 1000))
print(f"step 500: {lr[500]:.5f}, step 900: {lr[900]:.5f}")
print(f"share of the summed rate in the first half: {lr[:500].sum() / lr.sum():.2f}")

## Q5. Dropout at prediction time

A trained-or-not network has `Dropout(0.5)` after its hidden layer. You call `model(x, training=False)` twice, and `model(x, training=True)` twice, on the same input. Which pairs of outputs match?

Prediction: *(write here)*

In [ ]:
set_seed(SEED)
net = keras.Sequential(
    [
        keras.Input((5,)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(1),
    ]
)
x = rng.normal(size=(1, 5)).astype("float32")
off = [float(net(x, training=False)[0, 0]) for _ in range(2)]
on = [float(net(x, training=True)[0, 0]) for _ in range(2)]
print("training=False:", off, "| training=True:", on)

## Q6. L2 on a weight with no gradient

A weight starts at 1.0. The data loss does not depend on it, so its gradient is zero. The learning rate is 0.1 and the L2 strength $\lambda$ is 0.5, so each step is $w\leftarrow w-\eta\lambda w$. What is the weight after 10 steps, and after 100?

Prediction: *(write here)*

In [ ]:
w = 1.0
for step in range(1, 101):
    w -= 0.1 * 0.5 * w
    if step in (10, 100):
        print(f"after {step} steps: {w:.4f}")

## Q7. Sampling a learning rate

You want learning rates between 0.0001 and 0.1. You draw 1,000 values uniformly on that range. What fraction falls below 0.001? Then draw uniformly on the log scale instead. What fraction now?

Prediction: *(write here)*

In [ ]:
uniform = rng.uniform(1e-4, 1e-1, 1000)
log_uniform = 10 ** rng.uniform(-4, -1, 1000)
print(f"uniform: {(uniform < 1e-3).mean():.3f}, log-uniform: {(log_uniform < 1e-3).mean():.3f}")

## Answers

Check each against your prediction.

**Q1.** The model with no hidden layer scores 0.50 (it can get at most 3 of 4 right, because no straight line separates XOR). Four tanh units reach 1.00. The hidden layer bends the space so a line can finish the job.

**Q2.** Sigmoid: the first-layer gradient is about 3e-10 against 0.32 at the last layer, a ratio of 1e-9. ReLU is far better but not flat: 8e-6 against 2e-2, a ratio of 4e-4, because PyTorch's default initialisation is not He and shrinks the signal a little at each layer. Both vanish, and sigmoid is a hundred thousand times worse. Each sigmoid layer multiplies the gradient by a slope of at most 0.25.

**Q3.** SGD ends at about 1e-2 and Adam at about 4e-4, so Adam is lower by roughly a factor of 25. The steep $y$ direction forces SGD's small rate, so the flat $x$ direction contracts only by a factor of 0.981 per step (0.15 after 100 steps, which squares to the 1e-2 loss). Adam scales each direction separately and does not need the compromise.

**Q4.** The rate at step 500 is exactly half the peak, 0.005, and at step 900 it is 0.00024. The first half holds 82% of the summed rate, and the last 20% of steps hold 1.3%. That is why cosine schedules act like "train at a high rate, then fine-tune".

**Q5.** Both `training=False` calls match each other (-0.370 twice). The two `training=True` calls differ (0.366 and -1.735), because each draws a fresh random mask. Dropout is a training-only device, and the $1/(1-p)$ scaling is what lets the off state need no correction.

**Q6.** The weight shrinks by 0.95 each step: 0.599 after 10 steps and 0.006 after 100. L2 is called weight decay for this reason. Weights with no gradient decay to zero, so only weights the data keeps pushing back survive.

**Q7.** Uniform on [0.0001, 0.1] puts only about 0.8% of draws below 0.001, so it almost never tries the small rates. Uniform on the log scale puts about 33% there, since the range covers three decades and one of them is below 0.001. Sample scale-type hyperparameters on a log scale.